### Cell 1: widgets (plus a safety check)

In [0]:
dbutils.widgets.text(name="source_name", defaultValue="")
dbutils.widgets.text(name="file_format", defaultValue="")
dbutils.widgets.text(name="catalog", defaultValue="")
dbutils.widgets.text(name="on_incompatible",defaultValue=)

src_name     = dbutils.widgets.get("source_name")
file_format  = dbutils.widgets.get("file_format")
catalog_name = dbutils.widgets.get("catalog")
target_table = f"{catalog_name}.bronze.{src_name}_bronze"
# Stop early if a widget is empty or the format is unsupported
assert src_name and catalog_name, "source_name and catalog must be filled in"
assert file_format in ("csv", "json", "parquet"), f"Unsupported format: {file_format}"

### Cell 2: ingestion with rescue mode

In [0]:
import pyspark.sql.functions as F   # safer than "import *": it doesn't overwrite Python's sum/max/min

source_path          = f"/Volumes/olist_dev/bronze/raw_vol/{src_name}/"
checkpoint_path      = f"/Volumes/olist_dev/bronze/checkpoints/{src_name}/"
schema_location_path = f"/Volumes/olist_dev/bronze/checkpoints/{src_name}/schema/"

# Record the time BEFORE the stream starts. The monitoring step below
# only looks at rows written after this moment (this run's rows).
run_start = spark.sql("SELECT current_timestamp()").collect()[0][0]

if file_format == "csv":
    df = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", file_format)
        .option("cloudFiles.schemaLocation", schema_location_path)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaEvolutionMode", "rescue")
        .option("cloudFiles.rescuedDataColumn", "_rescued_data")
        .option("header", "true")
        .load(source_path))

elif file_format == "json":
    df = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", file_format)
        .option("cloudFiles.schemaLocation", schema_location_path)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaEvolutionMode", "rescue")
        .option("cloudFiles.rescuedDataColumn", "_rescued_data")
        .option("multiLine", "true")
        .load(source_path))

else:  # parquet
    df = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", file_format)
        .option("cloudFiles.schemaLocation", schema_location_path)
        .option("cloudFiles.schemaEvolutionMode", "rescue")
        .option("cloudFiles.rescuedDataColumn", "_rescued_data")
        .load(source_path))

df = (df
    .withColumn("ingestion_time", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_path")))   # which file each row came from

query = (df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    # mergeSchema removed on purpose: if the schema ever differs, the write fails loudly
    # instead of silently adding a column to Bronze.
    .outputMode("append")
    .trigger(availableNow=True)      # process everything new, then stop (good for hourly jobs)
    .toTable(target_table))

query.awaitTermination()
print(f" Ingestion finished for table: {target_table}")

### ## Cell 3: post-ingestion monitoring

In [0]:
FAIL_ON_INCOMPATIBLE = (
    dbutils.widgets.get("fail_on_incompatible").lower() == "true")# set True to make the job fail when existing columns break default it is false
    
IGNORED_KEYS = ["_file_path"]       # Auto Loader adds this key itself; it is not a data column

bronze_df  = spark.read.table(target_table)    # lazy: nothing is read yet
table_cols = set(bronze_df.columns)            # reading column names does not scan data we use the set to quickly check the col in the df does it already exist in the bronze table and compare it to the col give by the rescure and do furhter monitoring

if "_rescued_data" not in table_cols:
    print(f" No '_rescued_data' column in {target_table} yet.")
else:
    # Only this run's rows. Delta skips files whose ingestion_time is older than run_start,
    # so this does NOT scan the whole table.
    rescued_this_run = (bronze_df
        .filter(F.col("ingestion_time") >= F.lit(run_start))
        .filter(F.col("_rescued_data").isNotNull() & (F.trim(F.col("_rescued_data")) != ""))) # the trim is doing this here Original value	After trim
#       "hello"	"hello"
#       "  hello  "	"hello"

    rescued_count = rescued_this_run.count()

    if rescued_count == 0:
        print(f"🟢 Clean run. No rescued data in {target_table}.")
    else:
        # Turn the JSON text into a key->value map, list the keys, count rows per key.
        # Runs on the cluster over ALL rescued rows (no limit(5), no collect of raw rows).
        key_stats = (rescued_this_run
            .select(F.from_json("_rescued_data", "map<string,string>").alias("m")) # from jsonconvert the json string to spark maptype and "map<string,string>" tells Spark that both keys and values should be strings. spark convert those key value in the string
            .filter(F.col("m").isNotNull())                       # skips unparseable JSON safely
            .select(F.explode(F.map_keys("m")).alias("field"),    # one row per key F.map_keys("m") extracts only the keys: from the spark map stype that we convert and alias to m
                    F.col("m")["_file_path"].alias("sample_file")) # get the file path from the rescued data from spark map type and alias to sample_file
            .filter(~F.col("field").isin(*IGNORED_KEYS)) #.isin() accepst value like thi ising("x",("y")) so uncpack give us exactly that
            .groupBy("field")
            .agg(F.count("*").alias("row_count"), # row count per key we have extracted column from for evry record mean row and then the ol is repeated so we can calculate the count of the row and then we can get the first file path for sample checking
                 F.first("sample_file").alias("sample_file"))
            .collect())                                           # tiny: one row per field name After .collect(), key_stats becomes a Python list containing Spark Row objects.

        lower_map = {c.lower(): c for c in table_cols}
        new_cols, incompatible, case_variants = [], [], []

        for r in key_stats: #r = Row(field="salary", row_count=5, sample_file="file1.csv")
            f = r["field"]
            if f in table_cols:                 # exact name exists -> value did not fit the type
                incompatible.append(r)
            elif f.lower() in lower_map:        # same name, different upper/lower case
                case_variants.append(r)
            else:                               # name unknown to Bronze
                new_cols.append(r)

        def show(title, rows):
            if rows:
                print(f"\n{title}")
                for r in sorted(rows, key=lambda x: x["field"]): # x represent one row here and x["field"] is the field name
                    print(f"- {r['field']}  (rows: {r['row_count']}, e.g. {r['sample_file']})")

        print(f"\n⚠️ {rescued_count} rows with rescued data in this run ({target_table})")
        show("New columns detected:", new_cols)
        show("Incompatible schema changes detected:", incompatible)
        show("Column name case differences detected (check upstream naming):", case_variants)

        if FAIL_ON_INCOMPATIBLE and incompatible:
            raise Exception(f"Incompatible schema change in {target_table}: "
                            f"{[r['field'] for r in incompatible]}")